# CheXzero checkpoint visualization + CheXzero / ALBEF / BioViL-T comparison

This notebook produces:

1. **One PDF per CheXzero checkpoint**
   - Uses the same canonical image-label cases as the BioViL-T `same_50_albef_margin` output.
   - Each row: **Original + VinDr GT box | CheXzero map | CheXzero overlay + GT**.

2. **One final cross-model PDF**
   - Uses the exact same case order.
   - Each row: **Original + GT | CheXzero | ALBEF | BioViL-T**.
   - You explicitly choose the CheXzero checkpoint after inspecting the per-checkpoint PDFs.

The BioViL-T manifest is used as the canonical case list to prevent silent image/label mismatch across methods.

**Note:** the reference BioViL-T output normally contains 100 **image-label pairs** (50 Cardiomegaly + 50 Pleural effusion). The notebook prints the number of unique CXRs separately.


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

pd.set_option("display.max_columns", 100)

TARGET_LABELS = ["Cardiomegaly", "Pleural effusion"]
EXPECTED_PAIR_COUNT = 100
EXPECTED_PER_LABEL = 50

CASES_PER_PAGE = 5
OVERLAY_ALPHA = 0.50
CMAP_NAME = "magma"
FIG_DPI = 150
PDF_DPI = 150

SAVE_PNG_PAGES = False


## Paths and configuration


In [ ]:
# CheXzero output root from your SLURM extraction.
CHEXZERO_ROOT = Path(
    "/home/vault/iwi5/iwi5362h/chexzero_heatmaps"
)

# Reference BioViL-T output; also used as the canonical case list.
BIOVIL_OUTPUT_DIR = Path(
    "/home/vault/iwi5/iwi5362h/results/biovil_t_phrase_grounding/"
    "same_50_albef_margin"
)

# CHANGE THIS ONE to the ALBEF ITC-margin Grad-CAM directory
# containing {image_id}.pt files.
ALBEF_HEATMAPS_DIR = Path(
    "/CHANGE/THIS/TO/YOUR/ALBEF/ITC_MARGIN_HEATMAPS"
)

IMAGES_ROOT = Path(
    "/home/woody/iwi5/iwi5362h/data/vindr_cxr/test"
)

ANNOTATIONS_CSV = Path(
    "/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/"
    "annotations_test.csv"
)

IMAGE_METADATA_CSV = Path(
    "/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv"
)

VIS_OUTPUT_DIR = Path(
    "/home/vault/iwi5/iwi5362h/results/visualization/"
    "chexzero_albef_biovil_comparison"
)

# Leave None until you inspect the individual CheXzero checkpoint PDFs.
COMPARISON_CHEXZERO_CHECKPOINT = None

VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    CHEXZERO_ROOT,
    BIOVIL_OUTPUT_DIR,
    IMAGES_ROOT,
    ANNOTATIONS_CSV,
    IMAGE_METADATA_CSV,
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("CheXzero root:", CHEXZERO_ROOT)
print("BioViL-T root:", BIOVIL_OUTPUT_DIR)
print("ALBEF root:   ", ALBEF_HEATMAPS_DIR)
print("Outputs:      ", VIS_OUTPUT_DIR)


## Shared helpers


In [ ]:
def safe_torch_load(path):
    path = Path(path)
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def as_2d_numpy(value):
    if torch.is_tensor(value):
        value = value.detach().cpu().float().numpy()
    arr = np.asarray(value, dtype=np.float32).squeeze()
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D map, got shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError("Map contains NaN or infinity")
    return arr


def minmax_for_display(arr):
    arr = as_2d_numpy(arr)
    lo = float(arr.min())
    hi = float(arr.max())
    if hi - lo <= 1e-12:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - lo) / (hi - lo)).astype(np.float32)


def ensure_display_range(arr):
    arr = as_2d_numpy(arr)
    if float(arr.min()) >= -1e-6 and float(arr.max()) <= 1.0 + 1e-6:
        return np.clip(arr, 0, 1).astype(np.float32)
    return minmax_for_display(arr)


def resize_map(arr, size_wh):
    arr = ensure_display_range(arr)
    if arr.shape == (size_wh[1], size_wh[0]):
        return arr
    pil = Image.fromarray(np.round(arr * 255).astype(np.uint8), mode="L")
    pil = pil.resize(size_wh, Image.Resampling.BILINEAR)
    return np.asarray(pil, dtype=np.float32) / 255.0


def choose_column(df, candidates, purpose):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    raise KeyError(
        f"Cannot find {purpose}. Available columns: {list(df.columns)}"
    )


def load_image(image_id):
    path = IMAGES_ROOT / f"{image_id}.png"
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as handle:
        return handle.convert("RGB")


def make_overlay(image, heatmap, alpha=OVERLAY_ALPHA, valid_mask=None):
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    hm = resize_map(heatmap, image.size)

    if valid_mask is None:
        valid = np.ones_like(hm, dtype=np.float32)
    else:
        valid = np.asarray(valid_mask, dtype=np.float32)
        if valid.shape != hm.shape:
            pil = Image.fromarray(np.round(valid * 255).astype(np.uint8), mode="L")
            pil = pil.resize(image.size, Image.Resampling.NEAREST)
            valid = (np.asarray(pil, dtype=np.float32) > 127).astype(np.float32)

    color = colormaps[CMAP_NAME](np.clip(hm, 0, 1))[..., :3]
    alpha_map = (alpha * np.clip(hm, 0, 1) * valid)[..., None]
    return np.clip((1 - alpha_map) * rgb + alpha_map * color, 0, 1)


def sanitize_filename(value):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip())
    return value.strip("_") or "checkpoint"


## Load VinDr GT boxes and image dimensions


In [ ]:
ann_raw = pd.read_csv(ANNOTATIONS_CSV)
ann_map = {
    "image_id": choose_column(
        ann_raw, ["image_id", "imageid"], "annotation image ID"
    ),
    "class_name": choose_column(
        ann_raw, ["class_name", "class", "label"], "annotation class"
    ),
    "x_min": choose_column(ann_raw, ["x_min", "xmin", "x1"], "x_min"),
    "y_min": choose_column(ann_raw, ["y_min", "ymin", "y1"], "y_min"),
    "x_max": choose_column(ann_raw, ["x_max", "xmax", "x2"], "x_max"),
    "y_max": choose_column(ann_raw, ["y_max", "ymax", "y2"], "y_max"),
}
annotations_df = ann_raw[list(ann_map.values())].rename(
    columns={v: k for k, v in ann_map.items()}
)
annotations_df["image_id"] = annotations_df["image_id"].astype(str)

meta_raw = pd.read_csv(IMAGE_METADATA_CSV)
meta_map = {
    "image_id": choose_column(
        meta_raw, ["image_id", "imageid"], "metadata image ID"
    ),
    "width": choose_column(
        meta_raw,
        ["width", "image_width", "original_width", "dim1", "w"],
        "original width",
    ),
    "height": choose_column(
        meta_raw,
        ["height", "image_height", "original_height", "dim0", "h"],
        "original height",
    ),
}
metadata_df = meta_raw[list(meta_map.values())].rename(
    columns={v: k for k, v in meta_map.items()}
)
metadata_df["image_id"] = metadata_df["image_id"].astype(str)

if metadata_df["image_id"].duplicated().any():
    raise ValueError("Image metadata contains duplicate image IDs")

metadata_lookup = (
    metadata_df.set_index("image_id")[["width", "height"]].to_dict("index")
)


def get_boxes(image_id, label, displayed_size):
    subset = annotations_df[
        (annotations_df.image_id == str(image_id))
        & (annotations_df.class_name == label)
    ]
    if subset.empty:
        return []

    if str(image_id) not in metadata_lookup:
        raise KeyError(f"Missing original dimensions for {image_id}")

    original = metadata_lookup[str(image_id)]
    sx = displayed_size[0] / float(original["width"])
    sy = displayed_size[1] / float(original["height"])

    boxes = []
    for row in subset.itertuples(index=False):
        x1 = np.clip(float(row.x_min) * sx, 0, displayed_size[0])
        y1 = np.clip(float(row.y_min) * sy, 0, displayed_size[1])
        x2 = np.clip(float(row.x_max) * sx, 0, displayed_size[0])
        y2 = np.clip(float(row.y_max) * sy, 0, displayed_size[1])
        if x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))
    return boxes


def draw_boxes(ax, boxes, color="lime", linewidth=2):
    for x1, y1, x2, y2 in boxes:
        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor=color,
                linewidth=linewidth,
            )
        )


print(
    f"annotations={len(annotations_df):,}; "
    f"metadata={len(metadata_df):,}"
)


## Load the canonical 100 BioViL-T image-label pairs


In [ ]:
BIOVIL_MANIFEST_CSV = BIOVIL_OUTPUT_DIR / "manifest.csv"
if not BIOVIL_MANIFEST_CSV.is_file():
    raise FileNotFoundError(BIOVIL_MANIFEST_CSV)

biovil_manifest = pd.read_csv(BIOVIL_MANIFEST_CSV)

required = {"image_id", "label", "heatmap_path"}
missing = required - set(biovil_manifest.columns)
if missing:
    raise KeyError(
        f"BioViL-T manifest missing columns: {sorted(missing)}"
    )

biovil_manifest["image_id"] = biovil_manifest["image_id"].astype(str)
biovil_manifest = biovil_manifest[
    biovil_manifest["label"].isin(TARGET_LABELS)
].copy()

if biovil_manifest.duplicated(["image_id", "label"]).any():
    raise ValueError("Duplicate image-label pairs in BioViL-T manifest")


def resolve_biovil_path(saved_path):
    saved = Path(str(saved_path))
    if saved.is_file():
        return saved

    candidates = [
        BIOVIL_OUTPUT_DIR / "maps" / saved.name,
        BIOVIL_OUTPUT_DIR / saved.name,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate

    return candidates[0]


biovil_manifest["biovil_heatmap_path"] = (
    biovil_manifest["heatmap_path"].map(resolve_biovil_path)
)
biovil_manifest["image_path"] = biovil_manifest["image_id"].map(
    lambda x: IMAGES_ROOT / f"{x}.png"
)

label_order = {label: i for i, label in enumerate(TARGET_LABELS)}
cases_df = (
    biovil_manifest.assign(
        _label_order=biovil_manifest["label"].map(label_order)
    )
    .sort_values(["_label_order"], kind="stable")
    .drop(columns="_label_order")
    .reset_index(drop=True)
)

counts = cases_df["label"].value_counts().reindex(TARGET_LABELS, fill_value=0)

print("Canonical image-label pairs:", len(cases_df))
print("Unique CXRs:               ", cases_df["image_id"].nunique())
display(counts.rename("pairs").to_frame())

if len(cases_df) != EXPECTED_PAIR_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAIR_COUNT} image-label pairs, "
        f"found {len(cases_df)}"
    )

if not (counts == EXPECTED_PER_LABEL).all():
    raise ValueError(
        f"Expected {EXPECTED_PER_LABEL} per label, got {counts.to_dict()}"
    )

missing_images = [
    str(p) for p in cases_df["image_path"] if not Path(p).is_file()
]
missing_biovil = [
    str(p)
    for p in cases_df["biovil_heatmap_path"]
    if not Path(p).is_file()
]

if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} canonical images missing; "
        f"examples={missing_images[:5]}"
    )

if missing_biovil:
    raise FileNotFoundError(
        f"{len(missing_biovil)} BioViL-T maps missing; "
        f"examples={missing_biovil[:5]}"
    )

display(cases_df.head(10))


## Model-specific heatmap loaders


In [ ]:
def load_biovil_map(path):
    payload = safe_torch_load(path)

    if "similarity_map_vis" in payload:
        arr = ensure_display_range(payload["similarity_map_vis"])
        used_key = "similarity_map_vis"
    elif "similarity_map_raw_aligned" in payload:
        arr = minmax_for_display(payload["similarity_map_raw_aligned"])
        used_key = "similarity_map_raw_aligned"
    elif "similarity_map_raw" in payload:
        arr = minmax_for_display(payload["similarity_map_raw"])
        used_key = "similarity_map_raw"
    else:
        raise KeyError(
            f"{path}: no recognized BioViL-T map key; "
            f"available={list(payload.keys())}"
        )

    region = payload.get("valid_region", [16, 240, 16, 240])
    if len(region) != 4:
        raise ValueError(f"{path}: invalid valid_region={region}")

    y0, y1, x0, x1 = map(int, region)
    valid_mask = np.zeros_like(arr, dtype=np.float32)
    valid_mask[y0:y1, x0:x1] = 1.0

    return arr, valid_mask, payload, used_key


def find_albef_file(image_id):
    exact = ALBEF_HEATMAPS_DIR / f"{image_id}.pt"
    if exact.is_file():
        return exact

    matches = list(ALBEF_HEATMAPS_DIR.rglob(f"{image_id}.pt"))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f"Ambiguous ALBEF files for {image_id}: {matches[:10]}"
        )
    raise FileNotFoundError(
        f"No ALBEF .pt for {image_id} under {ALBEF_HEATMAPS_DIR}"
    )


def load_albef_map(image_id, label):
    path = find_albef_file(image_id)
    payload = safe_torch_load(path)

    if label not in payload:
        raise KeyError(
            f"{path}: label {label!r} missing; "
            f"available={list(payload.keys())}"
        )

    obj = payload[label]

    if torch.is_tensor(obj) or isinstance(obj, np.ndarray):
        arr = obj
        used_key = "<direct tensor>"
    elif isinstance(obj, dict):
        for key in (
            "cam_vis_up",
            "gradcam_vis_up",
            "cam_vis",
            "cam_positive_raw",
            "cam_raw",
        ):
            if key in obj:
                arr = obj[key]
                used_key = key
                break
        else:
            raise KeyError(
                f"{path}:{label}: no recognized ALBEF map key; "
                f"available={list(obj.keys())}"
            )
    else:
        raise TypeError(
            f"{path}:{label}: unsupported object type {type(obj)}"
        )

    return ensure_display_range(arr), payload, path, used_key


CHEXZERO_MAP_KEY_PRIORITY = (
    "cam_vis_up",
    "gradcam_vis_up",
    "heatmap_vis",
    "heatmap",
    "attention_gradcam_vis_up",
    "attention_gradcam",
    "gradcam",
    "cam_up",
    "cam_upsampled",
    "cam_vis",
    "cam",
    "map",
)


def label_slug(label):
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")


def find_chexzero_file(checkpoint_dir, image_id, label):
    checkpoint_dir = Path(checkpoint_dir)

    for candidate in [
        checkpoint_dir / f"{image_id}.pt",
        checkpoint_dir / "maps" / f"{image_id}.pt",
        checkpoint_dir / "heatmaps" / f"{image_id}.pt",
    ]:
        if candidate.is_file():
            return candidate

    exact_matches = list(checkpoint_dir.rglob(f"{image_id}.pt"))

    if len(exact_matches) == 1:
        return exact_matches[0]

    if len(exact_matches) > 1:
        preferred = [
            p
            for p in exact_matches
            if "maps" in p.parts or "heatmaps" in p.parts
        ]
        if len(preferred) == 1:
            return preferred[0]
        raise RuntimeError(
            f"Ambiguous CheXzero files for {image_id}: "
            f"{exact_matches[:10]}"
        )

    fuzzy = list(checkpoint_dir.rglob(f"{image_id}*.pt"))
    slug = label_slug(label)
    label_matches = [p for p in fuzzy if slug in p.stem.lower()]

    if len(label_matches) == 1:
        return label_matches[0]

    if len(fuzzy) == 1:
        return fuzzy[0]

    raise FileNotFoundError(
        f"No unambiguous CheXzero map for image={image_id}, "
        f"label={label!r}, checkpoint={checkpoint_dir}"
    )


def extract_2d_map_from_mapping(mapping, priorities, context):
    for key in priorities:
        if key in mapping:
            return mapping[key], key

    candidates = []
    for key, value in mapping.items():
        try:
            arr = as_2d_numpy(value)
            candidates.append((key, arr))
        except Exception:
            pass

    if len(candidates) == 1:
        key, arr = candidates[0]
        warnings.warn(
            f"{context}: using sole 2D field {key!r}. "
            "Add it to CHEXZERO_MAP_KEY_PRIORITY if this is the intended map."
        )
        return arr, key

    raise KeyError(
        f"{context}: no recognized map key. "
        f"available={list(mapping.keys())}; "
        f"2D candidates={[k for k, _ in candidates]}"
    )


def load_chexzero_map(checkpoint_dir, image_id, label):
    path = find_chexzero_file(checkpoint_dir, image_id, label)
    payload = safe_torch_load(path)

    obj = payload

    if isinstance(payload, dict):
        if label in payload:
            obj = payload[label]
        elif "heatmaps" in payload and isinstance(payload["heatmaps"], dict):
            if label in payload["heatmaps"]:
                obj = payload["heatmaps"][label]

    if torch.is_tensor(obj) or isinstance(obj, np.ndarray):
        arr = obj
        used_key = "<direct tensor>"
    elif isinstance(obj, dict):
        arr, used_key = extract_2d_map_from_mapping(
            obj,
            CHEXZERO_MAP_KEY_PRIORITY,
            f"{path}:{label}",
        )
    else:
        raise TypeError(
            f"{path}:{label}: unsupported CheXzero payload type {type(obj)}"
        )

    return ensure_display_range(arr), payload, path, used_key


## Discover CheXzero checkpoint output directories


In [ ]:
def discover_chexzero_checkpoints(root):
    root = Path(root)
    candidates = []

    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue

        if child.name.lower() in {
            "visualization",
            "visualizations",
            "pdfs",
            "comparison",
            "plots",
        }:
            continue

        if any(child.rglob("*.pt")):
            candidates.append(child)

    if not candidates and any(root.rglob("*.pt")):
        candidates = [root]

    if not candidates:
        raise RuntimeError(
            f"No CheXzero checkpoint directories containing .pt maps "
            f"found under {root}"
        )

    return candidates


CHEXZERO_CHECKPOINT_DIRS = discover_chexzero_checkpoints(CHEXZERO_ROOT)

checkpoint_table = pd.DataFrame(
    {
        "checkpoint_name": [p.name for p in CHEXZERO_CHECKPOINT_DIRS],
        "directory": [str(p) for p in CHEXZERO_CHECKPOINT_DIRS],
        "pt_files_recursive": [
            sum(1 for _ in p.rglob("*.pt"))
            for p in CHEXZERO_CHECKPOINT_DIRS
        ],
    }
)

print(
    f"Discovered {len(CHEXZERO_CHECKPOINT_DIRS)} "
    "CheXzero checkpoint outputs"
)
display(checkpoint_table)


## Validate all CheXzero checkpoints on the canonical pairs


In [ ]:
def validate_chexzero_checkpoint(checkpoint_dir, cases):
    records = []
    errors = []

    for row in cases.itertuples(index=False):
        try:
            arr, _, path, used_key = load_chexzero_map(
                checkpoint_dir,
                str(row.image_id),
                str(row.label),
            )

            records.append(
                {
                    "checkpoint": Path(checkpoint_dir).name,
                    "image_id": str(row.image_id),
                    "label": str(row.label),
                    "path": str(path),
                    "map_key": used_key,
                    "shape": str(tuple(arr.shape)),
                    "min": float(arr.min()),
                    "max": float(arr.max()),
                    "mean": float(arr.mean()),
                }
            )

        except Exception as exc:
            errors.append(
                {
                    "checkpoint": Path(checkpoint_dir).name,
                    "image_id": str(row.image_id),
                    "label": str(row.label),
                    "error": repr(exc),
                }
            )

    return pd.DataFrame(records), pd.DataFrame(errors)


coverage_rows = []
chexzero_validation = {}
chexzero_errors = {}

for checkpoint_dir in CHEXZERO_CHECKPOINT_DIRS:
    valid_df, error_df = validate_chexzero_checkpoint(
        checkpoint_dir,
        cases_df,
    )

    chexzero_validation[checkpoint_dir.name] = valid_df
    chexzero_errors[checkpoint_dir.name] = error_df

    coverage_rows.append(
        {
            "checkpoint": checkpoint_dir.name,
            "valid_pairs": len(valid_df),
            "missing_or_invalid_pairs": len(error_df),
            "map_keys": (
                ", ".join(sorted(valid_df["map_key"].astype(str).unique()))
                if not valid_df.empty
                else ""
            ),
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

bad = coverage_df[
    coverage_df["missing_or_invalid_pairs"] > 0
]

if not bad.empty:
    for checkpoint in bad["checkpoint"].tolist():
        print(f"\n--- {checkpoint} ---")
        display(chexzero_errors[checkpoint].head(10))

    raise RuntimeError(
        "At least one CheXzero checkpoint is missing canonical cases "
        "or has an unrecognized payload."
    )

print("All checkpoints cover every canonical image-label pair.")


## Inspect one CheXzero map before rendering all PDFs


In [ ]:
CHECKPOINT_INDEX = 0
CASE_INDEX = 0

checkpoint_dir = CHEXZERO_CHECKPOINT_DIRS[CHECKPOINT_INDEX]
row = cases_df.iloc[CASE_INDEX]

chex_map, chex_payload, chex_path, chex_key = load_chexzero_map(
    checkpoint_dir,
    row.image_id,
    row.label,
)

image = load_image(row.image_id)
boxes = get_boxes(row.image_id, row.label, image.size)
chex_up = resize_map(chex_map, image.size)

fig, axes = plt.subplots(
    1, 3, figsize=(12, 4), dpi=FIG_DPI
)

axes[0].imshow(image)
draw_boxes(axes[0], boxes)
axes[0].set_title(
    f"{CASE_INDEX+1:03d}. {row.image_id}\n"
    f"{row.label} | Original + GT"
)

axes[1].imshow(
    chex_up,
    cmap=CMAP_NAME,
    vmin=0,
    vmax=1,
)
axes[1].set_title(
    f"CheXzero map\n"
    f"{checkpoint_dir.name}\n"
    f"field={chex_key} | native={tuple(chex_map.shape)}"
)

axes[2].imshow(
    make_overlay(image, chex_up)
)
draw_boxes(axes[2], boxes)
axes[2].set_title("CheXzero overlay + GT")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Loaded:", chex_path)
if isinstance(chex_payload, dict):
    print("Top-level keys:", list(chex_payload.keys()))


## Generate one PDF per CheXzero checkpoint


In [ ]:
CHEXZERO_PDF_DIR = (
    VIS_OUTPUT_DIR / "chexzero_checkpoints"
)
CHEXZERO_PDF_DIR.mkdir(
    parents=True, exist_ok=True
)


def create_chexzero_checkpoint_page(
    checkpoint_dir,
    page_df,
    page_number,
    start_index,
):
    nrows = len(page_df)

    fig, axes = plt.subplots(
        nrows,
        3,
        figsize=(12.5, 3.45 * nrows),
        dpi=FIG_DPI,
        squeeze=False,
    )

    for r, row in enumerate(
        page_df.itertuples(index=False)
    ):
        image = load_image(row.image_id)
        boxes = get_boxes(
            row.image_id,
            row.label,
            image.size,
        )

        heatmap, _, _, used_key = load_chexzero_map(
            checkpoint_dir,
            row.image_id,
            row.label,
        )

        heatmap_up = resize_map(
            heatmap,
            image.size,
        )

        case_number = start_index + r + 1

        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], boxes)
        axes[r, 0].set_title(
            f"{case_number:03d}. {row.image_id}\n"
            f"{row.label} | Original + GT",
            fontsize=8,
        )

        axes[r, 1].imshow(
            heatmap_up,
            cmap=CMAP_NAME,
            vmin=0,
            vmax=1,
        )
        axes[r, 1].set_title(
            f"CheXzero normalized map\n"
            f"field={used_key} | native={tuple(heatmap.shape)}",
            fontsize=8,
        )

        axes[r, 2].imshow(
            make_overlay(
                image,
                heatmap_up,
            )
        )
        draw_boxes(axes[r, 2], boxes)
        axes[r, 2].set_title(
            "CheXzero overlay + GT",
            fontsize=8,
        )

        for ax in axes[r]:
            ax.axis("off")

    fig.suptitle(
        f"CheXzero attention Grad-CAM - "
        f"{checkpoint_dir.name} - page {page_number:02d}",
        fontsize=14,
        fontweight="bold",
        y=1.002,
    )

    plt.tight_layout()
    return fig


generated_checkpoint_pdfs = []

for checkpoint_dir in CHEXZERO_CHECKPOINT_DIRS:
    safe_name = sanitize_filename(
        checkpoint_dir.name
    )

    pdf_path = (
        CHEXZERO_PDF_DIR
        / f"chexzero_{safe_name}_all_100.pdf"
    )

    num_pages = math.ceil(
        len(cases_df) / CASES_PER_PAGE
    )

    print(
        f"\nRendering {checkpoint_dir.name}: "
        f"{len(cases_df)} pairs, {num_pages} pages"
    )

    with PdfPages(pdf_path) as pdf:
        for page_index in range(num_pages):
            start = (
                page_index * CASES_PER_PAGE
            )

            page_df = cases_df.iloc[
                start : start + CASES_PER_PAGE
            ]

            fig = create_chexzero_checkpoint_page(
                checkpoint_dir,
                page_df,
                page_index + 1,
                start,
            )

            pdf.savefig(
                fig,
                bbox_inches="tight",
                dpi=PDF_DPI,
            )

            if SAVE_PNG_PAGES:
                png_dir = (
                    CHEXZERO_PDF_DIR
                    / f"{safe_name}_pages"
                )
                png_dir.mkdir(
                    parents=True,
                    exist_ok=True,
                )
                fig.savefig(
                    png_dir
                    / f"page_{page_index+1:02d}.png",
                    dpi=FIG_DPI,
                    bbox_inches="tight",
                )

            plt.close(fig)

    generated_checkpoint_pdfs.append(
        {
            "checkpoint": checkpoint_dir.name,
            "pdf": str(pdf_path),
        }
    )

    print("Saved:", pdf_path)

checkpoint_pdf_df = pd.DataFrame(
    generated_checkpoint_pdfs
)
display(checkpoint_pdf_df)


## Choose the CheXzero checkpoint for the final comparison

Inspect the individual checkpoint PDFs, then set `COMPARISON_CHEXZERO_CHECKPOINT` to **one exact checkpoint name** below.

The notebook deliberately does not silently pick a checkpoint.


In [ ]:
print("Available checkpoint names:")
for checkpoint_dir in CHEXZERO_CHECKPOINT_DIRS:
    print(" -", checkpoint_dir.name)

print(
    "\nCurrent COMPARISON_CHEXZERO_CHECKPOINT:",
    COMPARISON_CHEXZERO_CHECKPOINT,
)

# Example:
# COMPARISON_CHEXZERO_CHECKPOINT = "your_checkpoint_directory_name"


## Validate ALBEF and BioViL-T coverage for the same pairs


In [ ]:
if not ALBEF_HEATMAPS_DIR.exists():
    raise FileNotFoundError(
        "Set ALBEF_HEATMAPS_DIR in the path cell first: "
        f"{ALBEF_HEATMAPS_DIR}"
    )

cross_model_records = []
cross_model_errors = []

for row in cases_df.itertuples(index=False):
    try:
        biovil_map, _, _, biovil_key = load_biovil_map(
            row.biovil_heatmap_path
        )

        albef_map, _, albef_path, albef_key = load_albef_map(
            row.image_id,
            row.label,
        )

        cross_model_records.append(
            {
                "image_id": row.image_id,
                "label": row.label,
                "albef_path": str(albef_path),
                "albef_key": albef_key,
                "albef_shape": str(tuple(albef_map.shape)),
                "biovil_path": str(row.biovil_heatmap_path),
                "biovil_key": biovil_key,
                "biovil_shape": str(tuple(biovil_map.shape)),
            }
        )

    except Exception as exc:
        cross_model_errors.append(
            {
                "image_id": row.image_id,
                "label": row.label,
                "error": repr(exc),
            }
        )

cross_model_validation_df = pd.DataFrame(
    cross_model_records
)
cross_model_errors_df = pd.DataFrame(
    cross_model_errors
)

print(
    f"valid={len(cross_model_validation_df)}, "
    f"errors={len(cross_model_errors_df)}"
)

if not cross_model_errors_df.empty:
    display(cross_model_errors_df.head(20))
    raise RuntimeError(
        "ALBEF/BioViL-T coverage is incomplete "
        "for the canonical pairs."
    )

display(cross_model_validation_df.head())


## Generate the final CheXzero / ALBEF / BioViL-T PDF


In [ ]:
if COMPARISON_CHEXZERO_CHECKPOINT is None:
    raise ValueError(
        "Set COMPARISON_CHEXZERO_CHECKPOINT after inspecting "
        "the individual CheXzero PDFs."
    )

checkpoint_lookup = {
    p.name: p
    for p in CHEXZERO_CHECKPOINT_DIRS
}

if COMPARISON_CHEXZERO_CHECKPOINT not in checkpoint_lookup:
    raise KeyError(
        f"Unknown checkpoint "
        f"{COMPARISON_CHEXZERO_CHECKPOINT!r}. "
        f"Available={list(checkpoint_lookup)}"
    )

comparison_checkpoint_dir = checkpoint_lookup[
    COMPARISON_CHEXZERO_CHECKPOINT
]

COMBINED_PDF_DIR = (
    VIS_OUTPUT_DIR / "cross_model"
)
COMBINED_PDF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

combined_pdf_path = (
    COMBINED_PDF_DIR
    / (
        "chexzero_"
        + sanitize_filename(
            COMPARISON_CHEXZERO_CHECKPOINT
        )
        + "_vs_albef_vs_biovil_t_all_100.pdf"
    )
)


def create_cross_model_page(
    page_df,
    page_number,
    start_index,
):
    nrows = len(page_df)

    fig, axes = plt.subplots(
        nrows,
        4,
        figsize=(16.0, 3.55 * nrows),
        dpi=FIG_DPI,
        squeeze=False,
    )

    for r, row in enumerate(
        page_df.itertuples(index=False)
    ):
        image = load_image(row.image_id)

        boxes = get_boxes(
            row.image_id,
            row.label,
            image.size,
        )

        chex_map, _, _, chex_key = load_chexzero_map(
            comparison_checkpoint_dir,
            row.image_id,
            row.label,
        )
        chex_up = resize_map(
            chex_map,
            image.size,
        )

        albef_map, _, _, albef_key = load_albef_map(
            row.image_id,
            row.label,
        )
        albef_up = resize_map(
            albef_map,
            image.size,
        )

        biovil_map, biovil_valid_mask, _, biovil_key = load_biovil_map(
            row.biovil_heatmap_path
        )
        biovil_up = resize_map(
            biovil_map,
            image.size,
        )

        case_number = (
            start_index + r + 1
        )

        axes[r, 0].imshow(image)
        draw_boxes(
            axes[r, 0],
            boxes,
        )
        axes[r, 0].set_title(
            f"{case_number:03d}. {row.image_id}\n"
            f"{row.label} | Original + GT",
            fontsize=8,
        )

        axes[r, 1].imshow(
            make_overlay(
                image,
                chex_up,
            )
        )
        draw_boxes(
            axes[r, 1],
            boxes,
        )
        axes[r, 1].set_title(
            "CheXzero\n"
            f"{COMPARISON_CHEXZERO_CHECKPOINT}\n"
            f"field={chex_key}",
            fontsize=8,
        )

        axes[r, 2].imshow(
            make_overlay(
                image,
                albef_up,
            )
        )
        draw_boxes(
            axes[r, 2],
            boxes,
        )
        axes[r, 2].set_title(
            "ALBEF ITC-margin Grad-CAM\n"
            f"field={albef_key}",
            fontsize=8,
        )

        axes[r, 3].imshow(
            make_overlay(
                image,
                biovil_up,
                valid_mask=biovil_valid_mask,
            )
        )
        draw_boxes(
            axes[r, 3],
            boxes,
        )
        axes[r, 3].set_title(
            "BioViL-T phrase grounding\n"
            f"field={biovil_key}\n"
            "valid=[16:240,16:240]",
            fontsize=8,
        )

        for ax in axes[r]:
            ax.axis("off")

    fig.suptitle(
        "CheXzero vs ALBEF vs BioViL-T - "
        f"page {page_number:02d}",
        fontsize=14,
        fontweight="bold",
        y=1.002,
    )

    plt.tight_layout()
    return fig


num_pages = math.ceil(
    len(cases_df) / CASES_PER_PAGE
)

print(
    f"Rendering {len(cases_df)} pairs "
    f"across {num_pages} pages"
)
print(
    "CheXzero checkpoint:",
    COMPARISON_CHEXZERO_CHECKPOINT,
)

with PdfPages(combined_pdf_path) as pdf:
    for page_index in range(num_pages):
        start = (
            page_index * CASES_PER_PAGE
        )

        page_df = cases_df.iloc[
            start : start + CASES_PER_PAGE
        ]

        fig = create_cross_model_page(
            page_df,
            page_index + 1,
            start,
        )

        pdf.savefig(
            fig,
            bbox_inches="tight",
            dpi=PDF_DPI,
        )

        if SAVE_PNG_PAGES:
            png_dir = (
                COMBINED_PDF_DIR / "pages"
            )
            png_dir.mkdir(
                parents=True,
                exist_ok=True,
            )
            fig.savefig(
                png_dir
                / f"page_{page_index+1:02d}.png",
                dpi=FIG_DPI,
                bbox_inches="tight",
            )

        plt.close(fig)

print("Saved:", combined_pdf_path)


## Save the exact case order and output summary


In [ ]:
order_path = (
    VIS_OUTPUT_DIR
    / "canonical_100_image_label_pair_order.csv"
)
cases_df.to_csv(
    order_path,
    index=False,
)

summary_rows = [
    {
        "output_type": "chexzero_checkpoint_pdf",
        "checkpoint": row["checkpoint"],
        "path": row["pdf"],
    }
    for row in generated_checkpoint_pdfs
]

if COMPARISON_CHEXZERO_CHECKPOINT is not None:
    summary_rows.append(
        {
            "output_type": "cross_model_pdf",
            "checkpoint": COMPARISON_CHEXZERO_CHECKPOINT,
            "path": str(combined_pdf_path),
        }
    )

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    VIS_OUTPUT_DIR / "pdf_outputs.csv"
)
summary_df.to_csv(
    summary_path,
    index=False,
)

print("Saved case order:", order_path)
print("Saved PDF summary:", summary_path)
display(summary_df)


## Interpretation notes

- The notebook uses **image-label pairs** as the localization unit because each pathology has its own map.
- It prints the number of unique CXRs separately, so 100 maps are not accidentally described as 100 unique images if the same CXR appears for both pathologies.
- CheXzero and ALBEF use saved normalized visualization fields when available; min-max normalization is only a fallback.
- BioViL-T preserves its central valid `[16:240,16:240]` region and leaves the excluded border uncolored.
- Every method uses the same VinDr GT-box scaling from `test_meta.csv`.
- The final comparison uses **one explicitly selected CheXzero checkpoint**. The visualization notebook does not perform checkpoint selection.
- Independently normalized maps are appropriate for qualitative spatial comparison but not for comparing attribution magnitude across methods or images.
